In [1]:
pip show chromadb


Name: chromadb
Version: 1.0.12
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: C:\Users\hp\AppData\Roaming\Python\Python312\site-packages
Requires: bcrypt, build, fastapi, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-instrumentation-fastapi, opentelemetry-sdk, orjson, overrides, posthog, pydantic, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

# Load your parquet file
df = pd.read_parquet("C:/Users/hp/Downloads/wiki_chunks_with_embeddings.parquet")

# Show sample
print(df.head())

                           group_id   section    chunk_id  \
0  00163fe7688e71ce06f495a6811fef71  FullText  FullText_1   
1  00163fe7688e71ce06f495a6811fef71  FullText  FullText_2   
2  00163fe7688e71ce06f495a6811fef71  FullText  FullText_3   
3  00163fe7688e71ce06f495a6811fef71  FullText  FullText_4   
4  004d14af54583f57c92178446dffb1b6  FullText  FullText_1   

                                          chunk_text  \
0  marieve herington born february 22, 1988 is a ...   
1  and members of the royal jelly orchestra, appe...   
2  relentless there s that word again optimism of...   
3  com e marieve herington facebook marieve herin...   
4  for other people with the same name, see rob b...   

                                                text     emb_0     emb_1  \
0  marieve herington born february 22, 1988 is a ... -0.071074 -0.060847   
1  and members of the royal jelly orchestra, appe... -0.041640 -0.074159   
2  relentless there s that word again optimism of... -0.052331 -0.03

In [9]:
from chromadb import PersistentClient

# Step 1: Create unique IDs
df['unique_id'] = df['group_id'] + "_" + df['chunk_id']

# Step 2: Prepare lists
ids = df['unique_id'].tolist()
documents = df['chunk_text'].tolist()


In [10]:
 #Extract embedding columns assuming 384 dimensions: emb_0 to emb_383
embedding_cols = [f'emb_{i}' for i in range(384)]
embeddings = df[embedding_cols].values.tolist()

# Step 3: Initialize Chroma client and collection
client = PersistentClient(path="chroma_store")
collection = client.get_or_create_collection(name="wiki_chunks")


In [11]:
# Step 4: Batch add function
def batch_iterable(iterable, batch_size):
    for i in range(0, len(iterable), batch_size):
        yield iterable[i:i + batch_size]

batch_size = 5000  # Adjust if needed, must be <= max batch size allowed

# Step 5: Add data in batches
for batch_ids, batch_docs, batch_embeds in zip(
    batch_iterable(ids, batch_size),
    batch_iterable(documents, batch_size),
    batch_iterable(embeddings, batch_size)
):
    collection.add(
        ids=batch_ids,
        documents=batch_docs,
        embeddings=batch_embeds
    )

print(f"Added {len(ids)} documents in batches of {batch_size} to ChromaDB.")


Added 10022 documents in batches of 5000 to ChromaDB.


In [12]:
print(f"Total items in collection: {collection.count()}")

# Retrieve a couple of items by ID
sample_ids = ids[:2]
result = collection.get(ids=sample_ids)
print(result)


Total items in collection: 10022
{'ids': ['00163fe7688e71ce06f495a6811fef71_FullText_1', '00163fe7688e71ce06f495a6811fef71_FullText_2'], 'embeddings': None, 'documents': ['marieve herington born february 22, 1988 is a canadian. marieve herington born february 22, 1988 age 37 oakville, ontario, canadal actress who has appeared in recurring roles on how met your mother, good luck charlie and ever after high. she provides the voice of tilly green on the disney channel show occupations actress singer big city greens as well as voicing animated lead characters in delilah julius and pearlie. in addition to that, she voiced years active 1996 present website marieveherington. com celestia ludenburg in the popular anime video game danganronpa trigger happy havoc, otter in franklin and fifi la fume in tiny toons looniversity. at the age of 12, she began singing in major public performances. since the age of 16, she has been fronting her own jazz ensembles. currently, she performs with the mariev

In [14]:
results = collection.query(
    query_texts=["Who is Marieve Herington?"],
    n_results=2
)

for i, doc in enumerate(results['documents'][0]):
    print(f" Result {i+1}:\n{doc}")


 Result 1:
com e marieve herington facebook marieve herington on twitter e marieve herington on instagram categories 1988 births living people actresses from los angeles actresses from oakville, ontario canadian child actresses canadian emigrants to the united states canadian film actresses canadian jazz singers canadian television actresses canadian video game actresses canadian voice actresses canadian women jazz singers franco ontarian people jazz musicians from california singers from los angeles university of toronto alumni 21st century canadian actresses 21st century canadian women singers
 Result 2:
and members of the royal jelly orchestra, appearing on two more albums. she sang the theme songs for cbc television s sesame park, nelvana s pippi longstocking and tvontario s marigold s mathemagics. herington moved to los angeles in 2008. 2i the marieve herington band performs in southern california and toronto. their latest album is midnight sessions. on april 14, 2021, herington a